# Aula 1B — Agentes e ferramentas controladas

## Caso prático: triagem de incidentes por ID de ticket

Neste notebook, você vai transformar um fluxo linear em um fluxo com decisão de uso de ferramenta.

### Objetivos de aprendizagem

Ao final, você será capaz de:
- explicar quando usar workflow linear e quando usar tool calling;
- definir uma ferramenta com contrato claro de entrada e saída;
- inspecionar o ciclo decisão -> execução de ferramenta -> resposta final.

### Resultado esperado

Você executará dois cenários:
1. modo demonstração (sem API), para visualizar o ciclo de ferramenta;
2. modo opcional com agente real, para comparar comportamento.

### Mapa do fluxo

```text
usuario -> agente -> decide usar ferramenta -> buscar_ticket -> agente -> resposta
```

## 1. Setup e modos de execução

### Objetivo
Executar o notebook em modo demonstração ou, opcionalmente, com agente real.

### Escolha do modo
- demonstração: não requer API e permite observar o ciclo com previsibilidade;
- real: requer `OPENAI_API_KEY`, `OPENAI_MODEL` e dependências opcionais.

### Critério de sucesso
A saída de configuração mostra claramente o modo ativo antes de você avançar.

In [14]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass, asdict
from typing import Any, Callable

from dotenv import load_dotenv

In [15]:
load_dotenv()

USE_DEMO_MODE = os.getenv("USE_DEMO_MODE")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Modo demonstração:", USE_DEMO_MODE)
print("Modelo configurado:", OPENAI_MODEL)

Modo demonstração: false
Modelo configurado: gpt-4.1-mini


## 2. Base de dados didática e funções puras

### Objetivo
Garantir que a consulta de tickets seja previsível e controlada.

### O que observar
- a função recebe entrada explícita (`ticket_id`);
- a saída é estruturada;
- não há escrita nem consulta arbitrária.

### Checkpoint
Teste `buscar_ticket` com ID válido e inválido e compare os dois formatos de retorno.

In [16]:
TICKETS = {
    "INC-2026-001": {
        "ticket_id": "INC-2026-001",
        "title": "Falhas de login após atualização",
        "service": "authentication",
        "description": "Clientes relatam erro 500 e timeout ao tentar entrar na plataforma.",
        "priority": "high",
        "status": "open",
    },
    "INC-2026-002": {
        "ticket_id": "INC-2026-002",
        "title": "Cobrança duplicada em pagamentos",
        "service": "billing",
        "description": "Alguns clientes identificaram duas cobranças para a mesma transação.",
        "priority": "medium",
        "status": "open",
    },
    "INC-2026-003": {
        "ticket_id": "INC-2026-003",
        "title": "Lentidão no relatório semanal",
        "service": "analytics",
        "description": "A geração do relatório semanal está mais lenta que o esperado, sem indisponibilidade.",
        "priority": "low",
        "status": "open",
    },
}


def normalizar_ticket_id(ticket_id: str) -> str:
    """Normaliza o identificador antes da consulta."""
    return ticket_id.strip().upper()


def ticket_nao_encontrado(ticket_id: str) -> dict[str, str]:
    """Representa uma falha de consulta como dado estruturado, não como texto solto."""
    return {
        "error": "ticket_not_found",
        "ticket_id": ticket_id,
    }


def buscar_ticket(ticket_id: str, tickets: dict[str, dict[str, str]] = TICKETS) -> dict[str, str]:
    """
    Recupera os campos autorizados de um ticket pelo identificador.

    Esta é a capacidade que será exposta ao agente. Ela não aceita SQL,
    filtros arbitrários ou comandos de escrita; aceita apenas um ticket_id.
    """
    ticket_id_normalizado = normalizar_ticket_id(ticket_id)
    ticket = tickets.get(ticket_id_normalizado)

    if ticket is None:
        return ticket_nao_encontrado(ticket_id_normalizado)

    return dict(ticket)


buscar_ticket("inc-2026-002")


{'ticket_id': 'INC-2026-002',
 'title': 'Cobrança duplicada em pagamentos',
 'service': 'billing',
 'description': 'Alguns clientes identificaram duas cobranças para a mesma transação.',
 'priority': 'medium',
 'status': 'open'}

## 3. Antes do agente: fluxo linear

### Objetivo
Revisar o baseline onde o código decide consultar o ticket antes da resposta.

### Quando esse padrão funciona melhor
Quando você já sabe quais dados sempre serão necessários.

### Critério de sucesso
Você identifica que a consulta ocorre automaticamente, sem decisão dinâmica no meio do ciclo.

In [17]:
def montar_contexto_deterministico(ticket_id: str) -> dict[str, Any]:
    ticket = buscar_ticket(ticket_id)

    return {
        "ticket_consultado_pelo_codigo": True,
        "ticket": ticket,
    }


montar_contexto_deterministico("INC-2026-001")


{'ticket_consultado_pelo_codigo': True,
 'ticket': {'ticket_id': 'INC-2026-001',
  'title': 'Falhas de login após atualização',
  'service': 'authentication',
  'description': 'Clientes relatam erro 500 e timeout ao tentar entrar na plataforma.',
  'priority': 'high',
  'status': 'open'}}

## 4. O que muda com ferramenta

### Objetivo
Entender a mudança central: quem decide consultar dados.

### Mudança de responsabilidade
No fluxo com ferramenta, o modelo solicita uma chamada estruturada; a aplicação valida e executa apenas o que é permitido.

### Critério de sucesso
Você consegue explicar o ciclo em 5 passos e apontar onde ocorre controle determinístico.

### Explicação detalhada da estrutura de Tool Calling desta seção

Esta célula de código cria um pequeno **protocolo interno** para representar chamadas de ferramenta de forma clara, previsível e auditável.

#### Ideia central

Quando um agente usa ferramenta, há dois momentos diferentes:
1. **pedido da ferramenta** (o agente decide o que quer consultar);
2. **resultado da ferramenta** (a aplicação executa e devolve dados).

As classes `ToolCall` e `ToolResult` existem para separar esses dois momentos com tipos explícitos.

#### O que `@dataclass(frozen=True)` faz

- `@dataclass` gera automaticamente métodos úteis como construtor e representação textual.
- `frozen=True` torna os objetos imutáveis após criação.
- Na prática, isso evita mutações acidentais durante o ciclo do agente (mais segurança e previsibilidade).

#### Papel de cada classe

1. `ToolCall`
- Representa a **intenção de chamada**.
- Campos:
  - `name`: qual ferramenta será chamada (ex.: `buscar_ticket`);
  - `arguments`: argumentos da chamada (ex.: `{"ticket_id": "INC-2026-001"}`).

2. `ToolResult`
- Representa o **retorno da execução**.
- Campos:
  - `name`: nome da ferramenta executada;
  - `output`: dados retornados pela função.

#### Por que não usar apenas dicionário solto

Daria para usar `dict`, mas essas classes ajudam em:
- legibilidade: fica óbvio se você está lidando com pedido ou resultado;
- manutenção: reduz erros de chave escrita errado;
- depuração: logs e rastreio ficam mais organizados;
- evolução: facilita adicionar validações sem quebrar o fluxo inteiro.

#### O que a função `descrever_ferramenta_buscar_ticket` faz

Ela cria os **metadados da ferramenta** para orientar o modelo e o orquestrador:
- `name`: identificador da ferramenta;
- `description`: quando usar;
- `input_schema`: formato permitido de entrada.

Esse schema é um limite de contrato: a chamada aceita apenas os campos esperados (aqui, `ticket_id`).

#### Fluxo completo desta parte

1. O sistema publica a descrição da ferramenta.
2. O agente decide se precisa da ferramenta e monta um `ToolCall`.
3. A aplicação valida/executa a ferramenta permitida.
4. O retorno é encapsulado em `ToolResult`.
5. O agente usa esse retorno para responder com base em dados reais.

Resumo: esta “bagunça” é uma forma explícita de manter controle, segurança e rastreabilidade no ciclo de tool calling.

In [18]:
@dataclass(frozen=True)
class ToolCall:
    name: str
    arguments: dict[str, str]


@dataclass(frozen=True)
class ToolResult:
    name: str
    output: dict[str, str]


def descrever_ferramenta_buscar_ticket() -> dict[str, Any]:
    """Metadados equivalentes aos que um framework fornece ao modelo."""
    return {
        "name": "buscar_ticket",
        "description": (
            "Recupera detalhes de um ticket de suporte a partir de seu identificador. "
            "Use quando for necessário conhecer título, serviço, descrição, prioridade ou status."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {
                    "type": "string",
                    "description": "Identificador do ticket, como INC-2026-001.",
                }
            },
            "required": ["ticket_id"],
            "additionalProperties": False,
        },
    }


descrever_ferramenta_buscar_ticket()


{'name': 'buscar_ticket',
 'description': 'Recupera detalhes de um ticket de suporte a partir de seu identificador. Use quando for necessário conhecer título, serviço, descrição, prioridade ou status.',
 'input_schema': {'type': 'object',
  'properties': {'ticket_id': {'type': 'string',
    'description': 'Identificador do ticket, como INC-2026-001.'}},
  'required': ['ticket_id'],
  'additionalProperties': False}}

## 5. Modo demonstração: ciclo do agente sem API

### Objetivo
Visualizar a mecânica de tool calling com regras transparentes.

### O que observar
- quando a ferramenta é solicitada;
- quando a aplicação executa a ferramenta;
- quando a resposta final é produzida.

### Checkpoint
O resultado deve mostrar uma chamada de ferramenta antes da resposta final.

In [ ]:
def extrair_ticket_id(texto: str) -> str | None:
    """Extrai um identificador no formato INC-AAAA-NNN de uma mensagem."""
    padrao = r"\bINC-\d{4}-\d{3}\b"
    correspondencia = re.search(padrao, texto.upper())
    return correspondencia.group(0) if correspondencia else None


def decidir_proxima_acao_demo(mensagem_usuario: str, contexto: list[dict[str, Any]]) -> ToolCall | None:
    """
    Simula a decisão do agente.

    Retorna uma ToolCall quando há um ticket mencionado e ainda não existe
    resultado de consulta no contexto. Caso contrário, indica que o ciclo
    pode terminar com uma resposta final.
    """
    ticket_id = extrair_ticket_id(mensagem_usuario)
    ja_consultou_ticket = any(item.get("type") == "tool_result" for item in contexto)

    if ticket_id and not ja_consultou_ticket:
        return ToolCall(name="buscar_ticket", arguments={"ticket_id": ticket_id})

    return None


def executar_tool_call(tool_call: ToolCall) -> ToolResult:
    """Executor controlado: apenas nomes de ferramentas previamente registrados são aceitos."""
    ferramentas: dict[str, Callable[..., dict[str, str]]] = {
        "buscar_ticket": buscar_ticket,
    }

    ferramenta = ferramentas.get(tool_call.name)
    if ferramenta is None:
        raise ValueError(f"Ferramenta não permitida: {tool_call.name}")

    return ToolResult(name=tool_call.name, output=ferramenta(**tool_call.arguments))


def gerar_resposta_final_demo(mensagem_usuario: str, contexto: list[dict[str, Any]]) -> str:
    """Gera uma resposta usando apenas resultados presentes no contexto."""
    resultados = [item["output"] for item in contexto if item.get("type") == "tool_result"]

    if not resultados:
        return "Não encontrei um ticket no contexto. Informe um identificador como INC-2026-001."

    ticket = resultados[-1]
    if ticket.get("error") == "ticket_not_found":
        return f"O ticket {ticket['ticket_id']} não foi encontrado."

    prioridade = ticket["priority"]
    recomendacao = (
        "Escalar para a equipe responsável pelo serviço."
        if prioridade in {"high", "critical"} or ticket["service"] == "billing"
        else "Registrar o acompanhamento e investigar dentro da fila normal."
    )

    return (
        f"Ticket {ticket['ticket_id']}: {ticket['title']}\n"
        f"Serviço: {ticket['service']} | Prioridade atual: {prioridade}\n"
        f"Resumo: {ticket['description']}\n"
        f"Recomendação: {recomendacao}"
    )


def executar_agente_demo(mensagem_usuario: str) -> dict[str, Any]:
    """Executa o ciclo: decidir → chamar ferramenta → receber resultado → responder."""
    contexto: list[dict[str, Any]] = []
    rastreio: list[dict[str, Any]] = []

    proxima_acao = decidir_proxima_acao_demo(mensagem_usuario, contexto)

    while proxima_acao is not None:
        rastreio.append({"step": "tool_requested", **asdict(proxima_acao)})
        resultado = executar_tool_call(proxima_acao)
        contexto.append({"type": "tool_result", "name": resultado.name, "output": resultado.output})
        rastreio.append({"step": "tool_completed", "name": resultado.name, "output": resultado.output})
        proxima_acao = decidir_proxima_acao_demo(mensagem_usuario, contexto)

    resposta = gerar_resposta_final_demo(mensagem_usuario, contexto)
    rastreio.append({"step": "final_response"})

    return {
        "response": resposta,
        "trace": rastreio,
    }


resultado_demo = executar_agente_demo("Analise o ticket INC-2026-002. Ele precisa ser escalado?")
resultado_demo


{'response': 'Ticket INC-2026-002: Cobrança duplicada em pagamentos\nServiço: billing | Prioridade atual: medium\nResumo: Alguns clientes identificaram duas cobranças para a mesma transação.\nRecomendação: Escalar para a equipe responsável pelo serviço.',
 'trace': [{'step': 'tool_requested',
   'name': 'buscar_ticket',
   'arguments': {'ticket_id': 'INC-2026-002'}},
  {'step': 'tool_completed',
   'name': 'buscar_ticket',
   'output': {'ticket_id': 'INC-2026-002',
    'title': 'Cobrança duplicada em pagamentos',
    'service': 'billing',
    'description': 'Alguns clientes identificaram duas cobranças para a mesma transação.',
    'priority': 'medium',
    'status': 'open'}},
  {'step': 'final_response'}]}

### Leitura do rastreio

Use o campo `trace` para validar o ciclo completo.

### Verificações esperadas
- existe uma etapa de solicitação da ferramenta;
- existe uma etapa de execução da ferramenta permitida;
- a resposta final usa apenas o contexto recuperado.

### Autoavaliação
Se o ticket não existir, você consegue localizar no rastreio onde essa informação apareceu?

## 6. Agente real com LangChain e `@tool`

### Objetivo
Comparar o modo demonstração com uma execução real de agente.

### Pré-requisitos
- variáveis `OPENAI_API_KEY` e `OPENAI_MODEL` configuradas;
- dependências opcionais instaladas.

### O que não muda
A regra de negócio continua na função funcional; o framework apenas expõe metadados da ferramenta ao modelo.

### Critério de sucesso
O agente chama a ferramenta quando necessário e responde sem inventar dados fora do contexto.

In [ ]:
!pip install langchain

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

@tool
def buscar_ticket_tool(ticket_id: str) -> dict[str, str]:
    """Recupera um ticket de suporte pelo identificador.

    Use esta ferramenta antes de analisar ou recomendar ações sobre um ticket
    específico. Ela retorna título, serviço, descrição, prioridade e status.
    """
    return buscar_ticket(ticket_id)

instrucoes_do_agente = """
Você é um agente de triagem de incidentes.

Regras:
- Quando o usuário mencionar um identificador de ticket, use buscar_ticket_tool
    antes de responder sobre o conteúdo, risco ou encaminhamento desse ticket.
- Não invente dados que não estejam na mensagem do usuário ou no resultado da ferramenta.
- Se a ferramenta informar ticket_not_found, comunique isso claramente.
- Você pode recomendar, mas não pode atualizar, apagar ou escalar tickets diretamente.
- Ao final, apresente: resumo, prioridade observada e recomendação objetiva.
"""

modelo = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
agente = create_agent(
    model=modelo,
    tools=[buscar_ticket_tool],
    system_prompt=instrucoes_do_agente,
)

resultado_real = agente.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Analise o ticket INC-2026-002. Ele precisa ser escalado?",
        }
    ]
})

print(resultado_real["messages"][-1].content)


Resumo: O ticket INC-2026-002 refere-se a um problema de cobrança duplicada em pagamentos, onde alguns clientes foram cobrados duas vezes pela mesma transação. O serviço afetado é o de billing (cobrança) e o status do ticket está aberto. A prioridade atual é média.

Prioridade observada: Média.

Recomendação objetiva: Considerando que o problema afeta diretamente a experiência dos clientes e envolve questões financeiras, é importante que o ticket seja tratado com atenção e resolvido rapidamente. A escalada pode ser recomendada se a equipe atual não tiver capacidade ou autoridade para resolver o problema de forma eficaz e rápida, especialmente para evitar impactos maiores na satisfação dos clientes e possíveis perdas financeiras.


### Inspecionando as etapas do agente real

Após executar `invoke`, revise a sequência de mensagens para confirmar o comportamento.

### O que validar
- presença de `tool_call` quando o usuário informa um ticket;
- uso do resultado da ferramenta na resposta final;
- ausência de afirmações não suportadas pelos dados retornados.

In [22]:
if "resultado_real" not in globals():
    print("Execute a célula anterior após configurar as dependências e as variáveis de ambiente.")
else:
    for indice, mensagem in enumerate(resultado_real["messages"]):
        print(f"\n--- Mensagem {indice} | {type(mensagem).__name__} ---")

        if getattr(mensagem, "tool_calls", None):
            print("Tool calls:", mensagem.tool_calls)

        if getattr(mensagem, "content", None):
            print("Conteúdo:", mensagem.content)



--- Mensagem 0 | HumanMessage ---
Conteúdo: Analise o ticket INC-2026-002. Ele precisa ser escalado?

--- Mensagem 1 | AIMessage ---
Tool calls: [{'name': 'buscar_ticket_tool', 'args': {'ticket_id': 'INC-2026-002'}, 'id': 'call_6OF3uYrtD6ODH8jQdVf8WOQX', 'type': 'tool_call'}]

--- Mensagem 2 | ToolMessage ---
Conteúdo: {"ticket_id": "INC-2026-002", "title": "Cobrança duplicada em pagamentos", "service": "billing", "description": "Alguns clientes identificaram duas cobranças para a mesma transação.", "priority": "medium", "status": "open"}

--- Mensagem 3 | AIMessage ---
Conteúdo: Resumo: O ticket INC-2026-002 refere-se a um problema de cobrança duplicada em pagamentos, onde alguns clientes foram cobrados duas vezes pela mesma transação. O serviço afetado é o de billing (cobrança) e o status do ticket está aberto. A prioridade atual é média.

Prioridade observada: Média.

Recomendação objetiva: Considerando que o problema afeta diretamente a experiência dos clientes e envolve questões 

## 7. Representação equivalente no Langflow

### Objetivo
Mapear a mesma arquitetura para um fluxo visual sem perder governança.

### Arquitetura mínima

```text
Chat Input -> Agent -> Chat Output
                ^
                | Tools
                |
         Buscar Ticket / MCP Tools
```

### Escolhas de implementação
- opção A: componente customizado (rápido para prototipar localmente);
- opção B: MCP Tools (melhor separação entre provedor e consumidor de ferramenta).

### Critério de sucesso
Você consegue explicar qual opção usaria para estudo local e qual usaria para evolução de produto.

### Passo a passo no canvas do Langflow

1. Adicione `Chat Input`, `Agent` e `Chat Output`.
2. Configure o modelo no `Agent`.
3. Adicione a fonte de ferramenta (componente customizado ou `MCP Tools`).
4. Habilite **Tool Mode** na fonte da ferramenta.
5. Conecte a saída de ferramenta na entrada **Tools** do `Agent`.
6. Use descrição explícita da ação (`buscar_ticket`) e quando acioná-la.
7. Teste com: `Analise o ticket INC-2026-001.`

### Resultado esperado
O agente consulta a ferramenta antes de responder sobre conteúdo e risco do ticket.

### Erro comum
Descrição vaga da ferramenta costuma reduzir a chance de chamada correta pelo agente.

## 8. Segurança e confiabilidade: limites do agente

### Princípio
Use sempre: `modelo sugere -> código valida -> sistema executa (ou pede aprovação humana)`.

### Capacidades que devem permanecer bloqueadas
- alterar prioridade/status automaticamente;
- executar ações de infraestrutura;
- consultar dados fora do escopo autorizado;
- expor campos confidenciais por padrão;
- aprovar ações irreversíveis sem revisão humana.

### Checkpoint
Ao final desta seção, você deve conseguir listar quais operações são leitura, sugestão e escrita com aprovação.

In [23]:
def validar_pedido_de_escalonamento(ticket_id: str, motivo: str) -> dict[str, str]:
    """Exemplo de guarda determinística antes de qualquer ação externa."""
    ticket = buscar_ticket(ticket_id)

    if ticket.get("error") == "ticket_not_found":
        return {"approved": "false", "reason": "ticket_not_found"}

    if len(motivo.strip()) < 20:
        return {
            "approved": "false",
            "reason": "justification_too_short",
        }

    return {
        "approved": "true",
        "reason": "eligible_for_human_review",
    }


validar_pedido_de_escalonamento(
    "INC-2026-002",
    "Há suspeita de impacto financeiro por cobrança duplicada.",
)


{'approved': 'true', 'reason': 'eligible_for_human_review'}

## 9. Comparação final: workflow linear vs agente com ferramenta

| Aspecto | Workflow linear | Agente com tool calling |
|---|---|---|
| Quem decide consultar? | código da aplicação | modelo, dentro das capacidades autorizadas |
| Momento da consulta | antes da chamada ao LLM | durante o ciclo de execução |
| Ordem das etapas | fixa | parcialmente dinâmica |
| Previsibilidade | maior | menor |
| Flexibilidade | menor | maior |
| Custo/latência | normalmente menores | podem aumentar |

### Síntese prática
- prefira workflow linear quando o caminho é conhecido;
- prefira agente com ferramenta quando a necessidade de consulta depende do contexto;
- em ambos os casos, mantenha contratos, validações e trilha de auditoria.


### Fechamento
Se você concluiu esta prática, já consegue projetar um ciclo básico de tool calling com limites claros de permissão e validação.